In [ ]:
import re
import matplotlib.pyplot as plt
import os
import numpy as np
from matplotlib.colors import ListedColormap

# Function to read the file and find blocks with the specified Z and T
def find_blocks(file_path):
    blocks = []
    file_recent_path = os.path.join(os.getcwd(), file_path)
    print(file_recent_path)
    with open(file_recent_path, 'r') as file:
        block = []
        z = None
        t = None
        graph_index = None
        num_elements = None
        block_mat = []
        for line in file:
            line = line.strip()
            if line.startswith("Z = "):
                if block:
                    block.pop(len(block)-1)
                    print(block[0])
                    blocks.append(block)
                z = float(line.split()[-1])
                t = float(file.readline().split()[-1])
                block = []
            elif line.startswith("It is the"):
                lline = line.split()
                graph_index = lline[3]
                graph_index = graph_index.replace('nd', '') 
                graph_index = int(graph_index)
            elif line.startswith("Number of elements in canon_config:"):
                num_elements = int(line.split()[-1])
                block.append([z, t, graph_index, num_elements])
            elif line.startswith("3") or line.startswith("2") or line.startswith("1") or line.startswith("0"):
                block_mat.append(line)
            elif line == "":
                block.append(block_mat)
                block_mat = []
        if block:
            block.pop(len(block)-1)
            print(block[0])
            blocks.append(block)
    return blocks

# Function to calculate the average number of elements in canon_config for the same Z and T
def calculate_average_elements(blocks, target_z, target_t):
    total_elements = 0
    count = 0
    for block in blocks:
        z, t, graph_index, num_elements = block[0]
        if z == target_z and t == target_t:
            total_elements += num_elements
            count += 1
    return total_elements / count if count > 0 else 0

# Function to generate thermal graphs for the matrices in the blocks
def generate_thermal_graphs(file_path, blocks, target_z, target_t):
    pic_path = './pic/' + file_path + 'and its pics'
    if not os.path.exists(pic_path):
        os.makedirs(pic_path)
    # set the colors 
    colors = ['black', 'blue', 'white', 'purple', 'red'] 
    custom_cmap = ListedColormap(colors)
    for block in blocks:
        z, t, graph_index, num_elements = block[0]
        if z == target_z and t == target_t:
            draw_block = block
            draw_block.pop(0)
            for i, mat in enumerate(draw_block):
                number = []
                for j, mat_row in enumerate(mat):
                    # label the sublattice of the occupied area 
                    number_row = list(map(int, mat_row.split()))
                    for k in range(len(number_row)):
                        if (number_row[k] == 1):
                            number_row[k] = 3 * ( (j + k) % 2) + 1 
                    number.append(number_row) 
                number_array = np.array(number)
                SIZE = len(number_row)  
                # Create a figure and a 300x300 grid of axes
                print("half done for index " + str(i))
                plt.figure(figsize=(20, 20), dpi=300)
                plt.imshow(number_array, cmap = custom_cmap, interpolation = 'nearest')
                plt.colorbar()  # Add a colorbar to the heatmap  
                plt.suptitle(f'Z={z}, T={t}, graph={graph_index}, config={num_elements}, index={i}')
                plt.xticks(np.arange(0, SIZE, SIZE/5))
                plt.yticks(np.arange(0, SIZE, SIZE/5))
                plt.savefig(pic_path+f"/Z={z}_T={t}_graph={graph_index}_config={num_elements}, index={i}.png")
                plt.close()
                
# Main function
def main(file_path, target_z, target_t):
    blocks = find_blocks(file_path)
    average_elements = calculate_average_elements(blocks, target_z, target_t)
    print(f"Average number of elements in canon_config for Z={target_z}, T={target_t}: {average_elements}")
    generate_thermal_graphs(file_path, blocks, target_z, target_t)

# Replace 'your_input_file.txt' with the path to your input file
# Replace 0.7 and 0.7 with the target Z and T values you want to search for
main('output_dynamics=100_ITER=10000000_Random-graph=2.txt', 0.9, 0.99)

## This is for the renormalized model

In [ ]:
import re
import matplotlib.pyplot as plt
import os
import numpy as np
from matplotlib.colors import ListedColormap

# Function to read the file and find blocks with the specified Z and T
def find_blocks(file_path):
    blocks = []
    file_recent_path = os.path.join(os.getcwd(), file_path)
    print(file_recent_path)
    with open(file_recent_path, 'r') as file:
        block = []
        z = None
        t = None
        E = None
        graph_index = None
        block_mat = []
        for line in file:
            line = line.strip()
            if line.startswith("Z = "):
                if block:
                    blocks.append(block)
                z = float(line.split()[-1])
                t = float(file.readline().split()[-1])
                E = float(file.readline().split()[-1])
                block = []
            elif line.startswith("It is the"):
                lline = line.split()
                graph_index = lline[3]
                graph_index = graph_index.replace('nd', '') 
                graph_index = int(graph_index)
                block.append([z, t, E, graph_index])
            elif line.startswith("-1") or line.startswith("1"):
                block_mat.append(line)
            elif line == "":
                block.append(block_mat)
                block_mat = []
        if block:
            blocks.append(block)
    return blocks



# Function to generate thermal graphs for the matrices in the blocks
def generate_thermal_graphs(file_path, blocks, target_z, target_t):
    pic_path = './pic/' + file_path + 'and its pics'
    if not os.path.exists(pic_path):
        os.makedirs(pic_path)
    # set the colors 
    print(pic_path)
    colors = ['black', 'red'] 
    custom_cmap = ListedColormap(colors)
    i = 0
    for block in blocks:
        z, t, E, graph_index = block[0]
        if z == target_z and t == target_t:
            i = i + 1
            mat = block[1]
            number = []
            for mat_row in mat:
                # label the sublattice of the occupied area 
                number_row = list(map(int, mat_row.split()))
                number.append(number_row) 
            number_array = np.array(number)
            SIZE = len(number_row)  
            # Create a figure and a 300x300 grid of axes
            print("half done for index " + str(i))
            plt.figure(figsize=(30, 30), dpi=300)
            plt.imshow(number_array, cmap = custom_cmap, interpolation = 'nearest', vmin=-1, vmax = 1)
            plt.colorbar()  # Add a colorbar to the heatmap  
            plt.suptitle(f'Z={z}, T={t}, graph={graph_index}, index={i}, The Avarage Energy is {E}')
            plt.xticks(np.arange(0, SIZE, SIZE/5))
            plt.yticks(np.arange(0, SIZE, SIZE/5))
            plt.savefig(pic_path+f"/Z={z}, T={t}, graph={graph_index}, index={i}, The avarage energy is {E}.png")
            plt.close()
                
# Main function
def main(file_path, target_z, target_t):
    blocks = find_blocks(file_path)
    generate_thermal_graphs(file_path, blocks, target_z, target_t)

# Replace 'your_input_file.txt' with the path to your input file
# Replace 0.7 and 0.7 with the target Z and T values you want to search for
main('output_renormalized_SIZE=100_ITER=10000_Random-Field=5_INITIAL_ITER=5_SIGMA=2.000000.txt', 0.5, 20.99)
